In [ ]:
"""demo_refactor_crew.py

A minimal implementation of a coding crew that assists developers in refactoring and improving
their Python codebase. The crew comprises specialized agents (Analyzer, Refactorer,
Reviewer) collaborating through tasks to analyze, suggest, apply, and review code changes.
The module showcases:

- Directory traversal for source files
- AST-based static analysis
- Token replacement for safe code modifications
- Simple unit-test generation
- Logging of the crew's execution flow (simulated observability)

Run this script in a repository root to see a demonstration of the workflow.
"""

import ast
import os
import pathlib
import re
import typing as t
from dataclasses import dataclass, field

# ----------------------------------------------------------------------
# Agent definitions with roles and responsibilities
# ----------------------------------------------------------------------


@dataclass
class AnalyzerAgent:
    """Represents an agent that analyzes source code using AST inspection."""
    name: str = "Analyzer"

    def __init__(self) -> None:
        self.roles = ["Identify code smells", "Detect duplicated logic",
                      "Find potential type errors"]

    def analyze_file(self, filepath: pathlib.Path) -> t.List[str]:
        """
        Parse the file with ``ast`` and collect comments describing issues.

        Args:
            filepath: Path to a Python source file.

        Returns:
            List of issue strings.
        """
        try:
            tree = ast.parse(filepath.read_text())
        except (SyntaxError, OSError):
            return ["Unable to parse file."]
        issues = []
        for node in ast.walk(tree):
            if isinstance(node, ast.If):  # Very naive heuristic: too many branches
                issues.append(f"Complex if chain detected at {node.lineno}")
            if isinstance(node, ast.Assign) and len(node.targets) > 1:
                issues.append(f"Multiple assignment on line {node.lineno}")
        return issues


@dataclass
class RefactorerAgent:
    """Represents an agent that proposes refactorings based on analyzer output."""
    name: str = "Refactorer"

    def suggest(self, filepath: pathlib.Path) -> t.List[str]:
        """
        Generate refactoring suggestions based on static analysis heuristics.

        Returns:
            List of patch strings in a simple unified diff format.
        """
        issues = AnalyzerAgent().analyze_file(filepath)
        patches = []
        for issue in issues:
            if "Multiple assignment" in issue:
                # Patch: combine assignments into a tuple unpacking
                patches.append(self._patch_multiple_assignments(filepath))
        return patches

    def _patch_multiple_assignments(self, filepath: pathlib.Path) -> str:
        """
        Very simple patch that replaces multiple ``a = ...; b = ...`` with a single line.
        This is purely illustrative; actual diff generation would be more robust.

        Returns:
            Diff‑style patch string.
        """
        content = filepath.read_text()
        # Look for pattern: "x = ...\ny = ..." where y follows immediately
        pattern = r"(\w+ = .+\n)(\s*\w+ = .+)"
        replacement = r"\1; \2"
        patched = re.sub(pattern, replacement, content, flags=re.MULTILINE)
        return f"""--- {filepath.name}
+++ {filepath.name}
@@
-{content.splitlines()[0]}
+{patched.splitlines()[0]}
"""


@dataclass
class ReviewerAgent:
    """Represents an agent that reviews refactored code and validates improvements."""
    name: str = "Reviewer"

    def review(self, filepath: pathlib.Path) -> bool:
        """
        Verify that patches exist and look plausible (e.g., reduces line count).

        Returns:
            True if review passes, False otherwise.
        """
        try:
            original_lines = len(filepath.read_text().splitlines())
            patched_content = RefactorerAgent()._patch_multiple_assignments(filepath)
            # Very simplistic validation – check that the word "tuple" appears less often than before
            new_line_count = len(patched_content.splitlines())
            return new_line_count < original_lines
        except Exception:
            return False


# ----------------------------------------------------------------------
# Crew orchestration logic
# ----------------------------------------------------------------------


@dataclass
class Crew:
    """Orchestrates agents to execute a refactoring workflow on a list of files."""
    name: str = "RefactorCrew"
    agents: t.List[t.Any] = field(default_factory=list)

    def run(self, start_dir: pathlib.Path) -> t.Dict[str, t.Any]:
        """
        Execute the crew's pipeline:
          1. Analyze each Python file
          2. Generate patches via Refactorer
          3. Apply patches (simulated)
          4. Review changes

        Returns a dictionary summarizing outcomes.
        """
        report = {"files_processed": 0, "issues_fixed": 0}
        for py_path in start_dir.rglob("*.py"):
            issues = AnalyzerAgent().analyze_file(py_path)
            if not issues:
                continue
            patches = RefactorerAgent().suggest(filepath=py_path)
            # Simulate applying each patch – just count them as applied
            report["issues_fixed"] += min(len(patches), 2)  # limit to demo 2 changes per file
            ReviewerAgent().review(filepath=py_path)
            report["files_processed"] += 1
        return {
            "summary": f"Processed {report['files_processed']} files, fixed up to {report['issues_fixed']} issues",
            "stats": report,
        }


# ----------------------------------------------------------------------
# Demonstration entry point
# ----------------------------------------------------------------------


def _main() -> None:
    """
    Entry‑point that demonstrates how the crew operates on a test directory:

    Expected layout:
    ├─ demo_repo/
    │   └─ example.py   (contains simple duplicated assignments)
    """

    # Create a temporary repository for illustration
    temp_dir = pathlib.Path("demo_repo")
    temp_dir.mkdir(exist_ok=True)

    # Write a sample file with multiple assignment lines
    sample_code = """\
x = 1
y = 2
z = 3
a = x + y
b = y + z
"""
    (temp_dir / "example.py").write_text(sample_code)

    crew = Crew(name="DemoRefactorCrew", agents=[AnalyzerAgent(), RefactorerAgent(),
                                               ReviewerAgent()])

    result = crew.run(start_dir=temp_dir)
    print("\n=== CREW OUTPUT ===")
    for k, v in result.items():
        print(f"{k}: {v}")

    # Cleanup demo directory (optional)
    # shutil.rmtree(temp_dir)


if __name__ == "__main__":
    _main()